# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SohailAkhtarChanna/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!git clone https://github.com/SohailAkhtarChanna/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 147, done.
remote: Counting objects: 100% (147/147), done.
remote: Compressing objects: 100% (103/103), done.
remote: Total 147 (delta 55), reused 94 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (147/147), 1.92 MiB | 8.01 MiB/s, done.
Resolving deltas: 100% (55/55), done.


In [2]:
import os

DATA_PATH = "/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"

print("Dataset exists:", os.path.exists(DATA_PATH))
print("Dataset path:", DATA_PATH)

Dataset exists: True
Dataset path: /content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv


In [3]:
import pandas as pd

DATA_PATH = "/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head()

Shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [4]:
# Check the main performance columns
target_columns = [
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "trend_pct"
]

df[target_columns].describe()

,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,trend_pct
count,30000.000000,30000.00000,30000.000000,29875.000000,30000.000000,26612.000000
mean,0.510733,16.34238,2.534520,18.212921,0.768196,-4.785969
std,3.279162,15.21679,8.310096,29.472768,7.429454,473.861780
min,0.000000,0.00000,0.000000,0.000000,0.000000,-100.000000
25%,0.000000,6.20000,0.000000,0.000000,0.000000,-62.600000
50%,0.070000,10.80000,0.000000,5.000000,0.000000,-33.500000
75%,0.290000,22.30000,1.350000,23.530000,0.000000,0.000000
max,100.000000,245.00000,100.000000,300.000000,300.000000,44900.000000


In [5]:
# Check how the categorical outcome columns are distributed

print("Impression tiers:")
print(df["impression_tier"].value_counts(dropna=False))

print("\nPosition tiers:")
print(df["position_tier"].value_counts(dropna=False))

print("\nTrend direction:")
print(df["trend_direction"].value_counts(dropna=False))

Impression tiers:
impression_tier
low          11248
moderate     10469
good          7205
excellent     1078
Name: count, dtype: int64

Position tiers:
position_tier
page_1      11814
striking     7304
page_3_5     7242
top_3        2321
deep         1319
Name: count, dtype: int64

Trend direction:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


In [6]:
# Target variable
y = df["impression_tier"]

# Features we will use for the model
feature_columns = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "content_age_days",
    "days_since_last_update",
    "avg_position",
    "ctr",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

X = df[feature_columns].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

X shape: (30000, 12)
y shape: (30000,)

Target distribution:
impression_tier
low          11248
moderate     10469
good          7205
excellent     1078
Name: count, dtype: int64


In [7]:
# Check missing values before handling them
print("Missing values before:")
print(X.isnull().sum())

# Fill missing numeric values with the median
X = X.fillna(X.median())

print("\nMissing values after:")
print(X.isnull().sum().sum())

Missing values before:
search_volume             2468
competition               2468
cpc                       2468
word_count                7699
char_count                7699
content_age_days             0
days_since_last_update       0
avg_position                 0
ctr                          0
engagement_rate              0
scroll_rate                125
ai_traffic_pct               0
dtype: int64

Missing values after:
0


In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

Training set: (24000, 12)
Test set: (6000, 12)


In [9]:
# ML-08 — Load the FlyRank starter dataset

import pandas as pd
import numpy as np

# Load the anonymized starter dataset
DATA_PATH = "/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [10]:
from sklearn.tree import DecisionTreeClassifier

# Create the Decision Tree model
model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

# Train the model
model.fit(X_train, y_train)

print("Decision Tree trained successfully!")

Decision Tree trained successfully!


In [11]:
from sklearn.metrics import accuracy_score, classification_report

# Make predictions on the test set
y_pred = model.predict(X_test)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

# Detailed evaluation
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.6935

Classification Report:
              precision    recall  f1-score   support

   excellent       0.00      0.00      0.00       215
        good       0.65      0.65      0.65      1441
         low       0.78      0.88      0.82      2250
    moderate       0.62      0.60      0.61      2094

    accuracy                           0.69      6000
   macro avg       0.51      0.53      0.52      6000
weighted avg       0.66      0.69      0.68      6000



/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [17]:
from sklearn.tree import export_text

# Display the learned decision rules
rules = export_text(
    model,
    feature_names=feature_columns
)

print(rules)

|--- ctr <= 0.00
|   |--- avg_position <= 11.25
|   |   |--- days_since_last_update <= 103.50
|   |   |   |--- content_age_days <= 382.00
|   |   |   |   |--- content_age_days <= 175.50
|   |   |   |   |   |--- class: low
|   |   |   |   |--- content_age_days >  175.50
|   |   |   |   |   |--- class: low
|   |   |   |--- content_age_days >  382.00
|   |   |   |   |--- content_age_days <= 452.50
|   |   |   |   |   |--- class: moderate
|   |   |   |   |--- content_age_days >  452.50
|   |   |   |   |   |--- class: low
|   |   |--- days_since_last_update >  103.50
|   |   |   |--- days_since_last_update <= 115.00
|   |   |   |   |--- search_volume <= 5.00
|   |   |   |   |   |--- class: moderate
|   |   |   |   |--- search_volume >  5.00
|   |   |   |   |   |--- class: low
|   |   |   |--- days_since_last_update >  115.00
|   |   |   |   |--- avg_position <= 8.95
|   |   |   |   |   |--- class: low
|   |   |   |   |--- avg_position >  8.95
|   |   |   |   |   |--- class: low
|   |--- avg

In [18]:
import pandas as pd

feature_importance = pd.DataFrame({
    "feature": feature_columns,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print(feature_importance)

                   feature  importance
8                      ctr    0.694862
9          engagement_rate    0.154108
7             avg_position    0.059207
5         content_age_days    0.038221
6   days_since_last_update    0.032787
10             scroll_rate    0.008471
3               word_count    0.008026
4               char_count    0.002172
0            search_volume    0.002147
1              competition    0.000000
2                      cpc    0.000000
11          ai_traffic_pct    0.000000


## 1. Method choice and why

I will use a **Decision Tree classifier** because it is easy to interpret and can turn the February performance signals into simple decision rules. This fits the content opportunity lane because the goal is to rank content items for review, not to make a black-box prediction. The model will be compared with my Week-4 baseline using the same data, metric, and evaluation setup.


In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-08 Section 1 — method check

from sklearn.tree import DecisionTreeClassifier

print("Method: Decision Tree Classifier")
print("Reason: interpretable rules and suitable for ranking content opportunities.")

Method: Decision Tree Classifier
Reason: interpretable rules and suitable for ranking content opportunities.


## 2. Split design

I will use a **client-grouped split** so that content from the same client does not appear in both training and testing data. This reduces the risk of the model learning client-specific patterns and gives a more honest estimate of how the model may perform on unseen clients. The split will use the same development data and label definition as the Week-4 baseline.


In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-08 Section 2 — split design

from sklearn.model_selection import GroupShuffleSplit

# Check the client grouping column
print("Client column:", "client_id")
print("Number of unique clients:", df["client_id"].nunique())
print("Number of rows:", len(df))

# Create a grouped 80/20 train-test split
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(df, groups=df["client_id"])
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))
print("Training clients:", train_df["client_id"].nunique())
print("Testing clients:", test_df["client_id"].nunique())

# Verify that no client appears in both sets
overlap = set(train_df["client_id"]) & set(test_df["client_id"])

print("Client overlap:", len(overlap))

Client column: client_id
Number of unique clients: 32
Number of rows: 30000
Training rows: 23837
Testing rows: 6163
Training clients: 25
Testing clients: 7
Client overlap: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-08 Section 3 — train + compare

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

# Target
target = "impression_tier"

# Features
feature_columns = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "content_age_days",
    "days_since_last_update",
    "avg_position",
    "ctr",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

# Create train/test features
X_train = train_df[feature_columns].copy()
X_test = test_df[feature_columns].copy()

y_train = train_df[target]
y_test = test_df[target]

# Fill missing values using training-set medians only
train_medians = X_train.median()

X_train = X_train.fillna(train_medians)
X_test = X_test.fillna(train_medians)

# Train Decision Tree
model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Evaluate
accuracy = accuracy_score(y_test, y_pred)

print("Decision Tree accuracy:", round(accuracy, 4))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    zero_division=0
))

Decision Tree accuracy: 0.6979

Classification Report:
              precision    recall  f1-score   support

   excellent       0.00      0.00      0.00       166
        good       0.67      0.41      0.51      1180
         low       0.80      0.90      0.84      2818
    moderate       0.57      0.65      0.60      1999

    accuracy                           0.70      6163
   macro avg       0.51      0.49      0.49      6163
weighted avg       0.68      0.70      0.68      6163



The Week-4 baseline is a transparent rule-based ranking approach that prioritizes pages with meaningful search exposure and CTR below the average for their position bucket. It assigns the `REVIEW_CTR` action and uses `high_volume_low_ctr` as the reason code.

For ML-08, I trained a Decision Tree classifier using the same content-performance dataset and a client-grouped 80/20 split. The Decision Tree achieved 69.79% accuracy on the grouped test set.

The two approaches serve as different forms of decision support: the Week-4 baseline ranks review opportunities using a transparent score, while the Decision Tree predicts the `impression_tier` class. Therefore, their raw outputs should not be treated as directly comparable metrics. The Decision Tree result provides a measured classification benchmark, while the Week-4 rule remains the transparent ranking baseline.


In [20]:
# Compare the model with the Week-4 baseline

print("Week-4 baseline:")
print("Rule: prioritize high-volume pages with CTR below the average for their position bucket.")
print("Action: REVIEW_CTR")
print("Reason code: high_volume_low_ctr")

print("\nML-08 Decision Tree:")
print("Target:", target)
print("Evaluation split: client-grouped 80/20 split")
print("Accuracy:", round(accuracy, 4))

Week-4 baseline:
Rule: prioritize high-volume pages with CTR below the average for their position bucket.
Action: REVIEW_CTR
Reason code: high_volume_low_ctr

ML-08 Decision Tree:
Target: impression_tier
Evaluation split: client-grouped 80/20 split
Accuracy: 0.6979


The Decision Tree performs best on the `low` class and has difficulty separating the higher-impression classes. In the grouped test set, it correctly classified 2,524 of 2,818 `low` items, but only 484 of 1,180 `good` items and 1,293 of 1,999 `moderate` items. It did not correctly classify any of the 166 `excellent` items.

The main source of error is confusion between neighboring classes. For example, 661 `good` items were predicted as `moderate`, while 603 `moderate` items were predicted as `low`. This suggests that the measured features do not clearly separate the higher tiers for unseen clients.

The model relies most strongly on CTR, which accounts for about 68.0% of the fitted tree's feature importance, followed by engagement rate at about 16.4%. Days since last update and average position have smaller importance. Search volume has zero importance in this fitted tree, even though search visibility was an important part of the Week-4 rule-based baseline.

Overall, the Decision Tree achieved 69.79% accuracy, but its macro F1 score was 0.49 and it missed the `excellent` class completely. Therefore, I would use the model as directional decision support rather than as an automatic ranking or approval system. The main improvement area is better separation of the minority `excellent` class and validation of whether the selected signals provide useful opportunity information beyond the transparent Week-4 baseline.


In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# SECTION 4: Feature importance

import pandas as pd

feature_importance = pd.DataFrame({
    "feature": feature_columns,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print("Feature importance:")
display(feature_importance)

Feature importance:


,feature,importance
8,ctr,0.679804
9,engagement_rate,0.163583
6,days_since_last_update,0.077895
7,avg_position,0.037702
10,scroll_rate,0.015309
5,content_age_days,0.015149
3,word_count,0.009093
4,char_count,0.001464
2,cpc,0.000000
1,competition,0.000000


In [22]:
# SECTION 4: Confusion matrix and class errors

from sklearn.metrics import confusion_matrix

labels = model.classes_

cm = confusion_matrix(y_test, y_pred, labels=labels)

cm_df = pd.DataFrame(
    cm,
    index=[f"Actual: {label}" for label in labels],
    columns=[f"Predicted: {label}" for label in labels]
)

print("Confusion Matrix:")
display(cm_df)

print("\nActual class counts:")
print(y_test.value_counts())

print("\nPredicted class counts:")
print(pd.Series(y_pred).value_counts())

Confusion Matrix:


,Predicted: excellent,Predicted: good,Predicted: low,Predicted: moderate
Actual: excellent,0,126,0,40
Actual: good,0,484,35,661
Actual: low,0,5,2524,289
Actual: moderate,0,103,603,1293



Actual class counts:
impression_tier
low          2818
moderate     1999
good         1180
excellent     166
Name: count, dtype: int64

Predicted class counts:
low         3162
moderate    2283
good         718
Name: count, dtype: int64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.